# 時系列・外部プレイヤー検証：データ準備と監査（学習なし）

対象：しょぼんｎ → ウルトラ立直。当該試合終了後の段位区分分類です。

ユーザー報告：台帳・特徴量11,573/6,053対局、同時間帯順序未確定4,467/3,389行、共通対局26行。H3で実際のキャッシュから再確認します。これらを本環境で実データ検証済みとは扱いません。
元ノートブックは変更せず、23特徴量の式も維持します。
**新ルートHは保存キャッシュ専用。H5の学習・評価は既定で無効、Optunaはありません。旧ZIP解析セルA4は今回実行しません。**

基準ソース：`yuukienomoto/report_-syobon_n` commit `4d7b41dae7901b08ef7f865b8591348f2cbed642`。
履歴値：基準版 ROC-AUC **0.8340**、外部 Accuracy **35.59%**（Balanced Accuracy **50.80%**）。
再計算した結果ではなく、比較用の保存値です。

今回の実行順：**セル1 → セル2 → セル3 → 追加A1 → 追加A2 → H1〜H4**。旧ルートのセル4監査は厳密順序・全履歴を要求するため変更せず保持しています。
不足や不確実性があれば表とCSVを出して停止します。入力を直したらセル3から再実行してください。
この段階で作るのは学習可能性を検査した入力行列であり、性能の改善を保証するものではありません。


## 入力データの契約（牌譜再解析は自動実行しません）

各プレイヤーについて2表を用意します。CSVまたはParquet、または明示的に指定したDataFrameを使用可能です。

|表|必須列|条件|
|---|---|---|
|manifest|`file_id`, `dan`, `game_order`|本人の全有効対局を段位17・初期期間も含めて列挙。`game_order`は初回からの1始まり連番。段位列名は設定で変更可|
|features|`file_id` と基本18特徴量|**補完・平滑化前**の値。非発生事象のNaNは保持。既存の補完済み`micro_dataset_df`は使用不可|

`micro_features_df`（しょぼんn）と`ultra_micro_features_df`（外部）は再利用候補です。
ただし前者は従来の6,015試合だけなので、段位17等の履歴が不足する可能性があります。
manifestを抽出済み特徴量の行だけから作ると、欠落した試合を検出できません。
`model_target_df` / 評価用`ultra_target_df`も全対局台帳の代わりにはなりません。

既存Colabのメモリに残っていれば、ノートブックを保存変更せず一時セルで例えば
`micro_features_df.to_csv('/content/drive/MyDrive/卒論/validation_inputs/shobon_raw.csv', index=False)`
と保存して再利用できます（保存先フォルダを先に作成）。すでに保存済みならそのファイルを指定します。
全対局台帳の段位は`dan_map`や全件の`dan_df`が候補ですが、**時系列と完全性は別途確認が必要**です。
未保存の場合は追加A1〜A5でZIPから生成できます。ZIPの解析は明示的な実行セルに限定します。

対局IDの日時は通常「時」単位です。同一時間内のハッシュ順を真の対局順とみなしません。
`game_order`の順序根拠と、全対局台帳の完全性・元特徴量であることを確認した場合にだけ、セル2の確認フラグをTrueにしてください。
単に検査を通すためのフラグ変更は不可です。証拠がなければ監査で停止するのが正常です。


## セル1：Google Drive接続


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## セル2：入力設定（ここを編集）


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json, re, hashlib, uuid
from datetime import datetime, timezone
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/卒論')
REPORT_ROOT = ROOT / 'validation_reports'
WINDOW = 10
INITIAL_EXCLUDE = 1500

SOURCES = {
    'shobon': {
        'manifest_path': ROOT / 'validation_inputs/shobon_manifest.csv',
        'features_path': ROOT / 'validation_inputs/shobon_raw.csv',
        'dan_column': 'dan',  # 保存データがshobon_danなら変更
        'raw_features_confirmed': False,
        'full_history_confirmed': False,
        'order_verified': False,
        'order_evidence': '',  # 同一時間内も含む順序を確認した情報源
        'history_evidence': '',  # 本人の初回からの全有効対局と確認した根拠
        'features_evidence': '',  # 抽出関数の版と補完前であることの根拠
    },
    'ultra': {
        'manifest_path': ROOT / 'validation_inputs/ultra_manifest.csv',
        'features_path': ROOT / 'validation_inputs/ultra_raw.csv',
        'dan_column': 'dan',  # 保存データがultra_danなら変更
        'raw_features_confirmed': False,
        'full_history_confirmed': False,
        'order_verified': False,
        'order_evidence': '',
        'history_evidence': '',
        'features_evidence': '',
    },
}
# 任意：同一ランタイムにあるDataFrameを明示指定。指定時は該当ファイルを読まない。
# 例 MEMORY_INPUTS = {('shobon', 'features'): micro_features_df}
# manifestも ('shobon', 'manifest') のキーで指定可能。自動推測はしません。
MEMORY_INPUTS = {}

REFERENCE_RESULTS = pd.DataFrame([
    {'source': '基準版・従来ランダムCV', 'metric': 'ROC-AUC', 'value': 0.8340},
    {'source': 'ウルトラ立直・従来外部評価', 'metric': 'Accuracy', 'value': 0.3559},
    {'source': 'ウルトラ立直・従来外部評価', 'metric': 'Balanced Accuracy', 'value': 0.5080},
])
# 将来の学習条件を記録するだけ。今回この設定で学習しません。
MODEL_PLAN = dict(objective='binary', class_weight='balanced', random_state=42,
                  n_estimators=200, learning_rate=0.03, max_depth=5,
                  subsample=0.8, colsample_bytree=0.8, verbosity=-1)
THRESHOLD_PLAN = 0.5
state = {}  # 設定変更時は旧検査結果を無効化
display(REFERENCE_RESULTS)


## セル3：共通関数の定義（学習・予測共通）

既存の定義上の注意：ノーテンは「門前かつ未リーチ」、無筋は実質非現物、良形は牌種数による代理指標。
副露復号・複数リーチへの安全判定にも制約があります。流局ノーテン率はテンパイ率の補数です。
クロス特徴量は試合単位の積・差で、同一局面の条件付き割合ではありません。**今回は変更しません。**

全台帳の通算1501試合目以降を候補とし、段位17も履歴と学習区間の補完平均に含めます。
履歴用には最初の採点候補の前9試合から必要です（通常1492試合目以降）。
特徴量のNaNと「特徴量行そのものの欠落」は区別し、行欠落は平均補完しません。


In [ ]:
BASE_FEATURES = [
    'first_discard_raw_unsafe_rate', 'first_discard_safe_rate',
    'early_noten_push_rate', 'mid_noten_push_rate', 'late_noten_push_rate',
    'las_desperate_push_rate', 'top_conservative_push_rate', 'flat_situation_push_rate',
    'riichi_dora_mean', 'chasing_riichi_good_shape_rate', 'chasing_riichi_low_value_rate',
    'chasing_riichi_rate', 'early_meld_ratio', 'late_meld_ratio', 'meld_dora_ratio',
    'raw_unsafe_discard_rate', 'ryuukyoku_tenpai_rate', 'ryuukyoku_noten_safe_rate',
]
CROSS_FEATURES = ['top_late_noten_push_rate', 'las_early_meld_rate',
                  'early_meld_with_dora_rate', 'chasing_riichi_quality_diff',
                  'flat_first_discard_safe_rate']
FEATURES = BASE_FEATURES + CROSS_FEATURES

class ValidationBlocked(ValueError):
    pass

def require(condition, message):
    if not condition:
        raise ValidationBlocked(message)

def config_signature():
    return json.dumps({'sources': SOURCES, 'window': WINDOW, 'initial': INITIAL_EXCLUDE,
                       'memory_keys': sorted(map(str, MEMORY_INPUTS))},
                      default=str, sort_keys=True, ensure_ascii=False)

def canonical_game_id(value):
    # パス・視点tw・拡張子を除去。同じ対局の別視点も同じIDにする。
    name = str(value).replace('\\', '/').rsplit('/', 1)[-1].lower()
    match = re.fullmatch(r'(\d{10}gm-[0-9a-f]+-[0-9a-f]+-[0-9a-f]+)(?:&tw=[0-3])?(?:\.mjlog)?', name)
    if not match:
        raise ValidationBlocked(f'解釈できない対局ID: {value}')
    return match.group(1)

def load_table(player, kind):
    if (player, kind) in MEMORY_INPUTS:
        df = MEMORY_INPUTS[(player, kind)].copy(deep=True)
        require(isinstance(df, pd.DataFrame), '入力はDataFrameで指定してください')
        origin = 'explicit in-memory DataFrame'
    else:
        path = Path(SOURCES[player][kind + '_path'])
        require(path.is_file(), f'未発見: {path}')
        if path.suffix.lower() == '.csv':
            df = pd.read_csv(path)
        elif path.suffix.lower() == '.parquet':
            df = pd.read_parquet(path)
        else:
            raise ValidationBlocked('CSV/Parquetのみ対応。pickleや牌譜は読みません')
        origin = str(path)
    require(df.columns.is_unique, '列名が重複しています')
    digest = hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()
                            + repr(list(df.columns)).encode()).hexdigest()
    return df, {'player': player, 'kind': kind, 'origin': origin,
                'rows': len(df), 'content_sha256': digest}

def add_cross(base):
    # 元ノートブックの5式をそのまま維持。局面別再集計はしない。
    x = base[BASE_FEATURES].copy()
    x['top_late_noten_push_rate'] = x['top_conservative_push_rate'] * x['late_noten_push_rate']
    x['las_early_meld_rate'] = x['las_desperate_push_rate'] * x['early_meld_ratio']
    x['early_meld_with_dora_rate'] = x['early_meld_ratio'] * x['meld_dora_ratio']
    x['chasing_riichi_quality_diff'] = x['chasing_riichi_good_shape_rate'] - x['chasing_riichi_low_value_rate']
    x['flat_first_discard_safe_rate'] = x['first_discard_safe_rate'] * (1.0 - x['flat_situation_push_rate'])
    return x[FEATURES]

def fit_imputation(training_rows):
    # 明示された学習区間だけ。外部データや検証区間は渡さない。
    require(len(training_rows) > 0, '補完平均の学習対象が空です')
    means = training_rows[BASE_FEATURES].mean()
    require(np.isfinite(means.to_numpy()).all(),
            '学習区間で全欠損の特徴量があります: ' + str(means.index[means.isna()].tolist()))
    return means

def transform_block(rows, means):
    require(list(means.index) == BASE_FEATURES, '補完値の列順が不正です')
    require(np.isfinite(means.to_numpy()).all(), '補完値に非有限値があります')
    require(len(rows) >= WINDOW, '10試合の履歴を作れません')
    require(rows['game_order'].diff().iloc[1:].eq(1).all(), '移動平均区間に対局の欠落があります')
    require(rows['game_id'].is_unique, '移動平均区間の対局IDが重複しています')
    filled = rows[BASE_FEATURES].fillna(means)
    require(np.isfinite(filled.to_numpy()).all(), '補完後に欠損・無限大が残っています')
    # 計算自体は既存のmin_periods=1を維持。ただし10試合未満は採点しない。
    rolled = add_cross(filled).rolling(window=WINDOW, min_periods=1).mean()
    meta = rows[['file_id', 'game_id', 'game_order', 'dan', 'target_label']].copy()
    ids = rows['game_id'].tolist()
    meta['source_ids'] = [tuple(ids[max(0, i-WINDOW+1):i+1]) for i in range(len(ids))]
    meta['full_window'] = [len(v) == WINDOW for v in meta['source_ids']]
    mask = meta['full_window'] & (meta['game_order'] > INITIAL_EXCLUDE) & meta['target_label'].notna()
    return {'X': rolled.loc[mask, FEATURES].reset_index(drop=True),
            'y': meta.loc[mask, 'target_label'].astype(int).reset_index(drop=True),
            'meta': meta.loc[mask].reset_index(drop=True)}

def source_ids(prepared):
    return {gid for window in prepared['meta']['source_ids'] for gid in window}

def fold_boundaries(n):
    # 初期1500試合を除いた実際の連続対局数で50%, 2/3, 5/6, 100%。段位で除外しない。
    remaining = n - INITIAL_EXCLUDE
    require(remaining >= 60, '初期期間後が60試合未満で3分割の履歴を確保できません')
    return [INITIAL_EXCLUDE + int(remaining * fraction) for fraction in (0.5, 2/3, 5/6, 1)]

def make_folds(data):
    boundaries = fold_boundaries(int(data['game_order'].max()))
    plans = []
    for k, (end, next_end) in enumerate(zip(boundaries[:-1], boundaries[1:]), 1):
        # trainには初期期間の末尾9試合の履歴を含める。
        train = data[data['game_order'] <= end].copy()
        validation = data[(data['game_order'] > end) & (data['game_order'] <= next_end)].copy()
        plans.append((k, train, validation))
    return plans

def audit_player(player, manifest, features, cfg):
    problems, tables = [], {}
    require('file_id' in manifest and 'file_id' in features, '両表にfile_idが必要です')
    require('game_order' in manifest and cfg['dan_column'] in manifest,
            '全対局台帳にgame_orderと指定した段位列が必要です')
    m, f = manifest.copy(), features.copy()
    m['game_id'] = m['file_id'].map(canonical_game_id)
    f['game_id'] = f['file_id'].map(canonical_game_id)
    for label, table in [('manifest', m), ('features', f)]:
        duplicate = table[table['game_id'].duplicated(keep=False)]
        tables[label + '_duplicates'] = duplicate
        if len(duplicate):
            problems.append(f'{label}: 正規化対局ID重複 {len(duplicate)}行（自動削除しません）')
    missing_cols = [c for c in BASE_FEATURES if c not in f]
    if missing_cols:
        problems.append('不足特徴量列: ' + str(missing_cols))
    m['game_order'] = pd.to_numeric(m['game_order'], errors='coerce')
    good_order = (m['game_order'].notna().all() and
                  np.array_equal(np.sort(m['game_order'].to_numpy()), np.arange(1, len(m)+1)))
    if not good_order:
        problems.append('game_orderが初回からの1始まり重複なし連番ではありません')
    m = m.sort_values('game_order', kind='stable').reset_index(drop=True)
    m['id_hour'] = pd.to_datetime(m['game_id'].str[:10], format='%Y%m%d%H', errors='coerce')
    if m['id_hour'].isna().any() or not m['id_hour'].is_monotonic_increasing:
        problems.append('対局IDの日時が不正、またはgame_orderと逆転しています')
    tables['same_hour'] = m[m['id_hour'].duplicated(keep=False)][['file_id', 'game_order', 'id_hour']]
    if not (cfg['order_verified'] and str(cfg['order_evidence']).strip()):
        problems.append('厳密な対局順序の保証・根拠が未確認（IDは時単位）')
    if not (cfg['full_history_confirmed'] and str(cfg['history_evidence']).strip()):
        problems.append('初回からの全有効対局台帳の完全性が未確認')
    if not (cfg['raw_features_confirmed'] and str(cfg['features_evidence']).strip()):
        problems.append('特徴量が補完・平滑化前で元定義と同一か未確認')
    m['dan'] = pd.to_numeric(m[cfg['dan_column']], errors='coerce')
    invalid_dan = m['dan'].isna() | ~np.isfinite(m['dan']) | (m['dan'] % 1 != 0) | ~m['dan'].between(0, 20)
    tables['invalid_dan'] = m.loc[invalid_dan, ['file_id', 'game_order', 'dan']]
    if (invalid_dan & (m['game_order'] > INITIAL_EXCLUDE)).any():
        problems.append('採点候補期間に欠損・不正段位があります')
    m['target_label'] = np.select([m['dan'] >= 18, m['dan'] <= 16], [1, 0], default=np.nan)
    # 初回採点1501試合目の窓は1492〜1501試合目。台帳自体は初回から必要。
    needed = m[m['game_order'] >= INITIAL_EXCLUDE - WINDOW + 2].copy()
    tables['missing_games'] = needed[~needed['game_id'].isin(f['game_id'])][['file_id', 'game_id', 'game_order', 'dan']]
    tables['extra_features'] = f.loc[~f['game_id'].isin(m['game_id']), ['file_id', 'game_id']]
    if len(tables['missing_games']):
        problems.append(f"必要な特徴量行が{len(tables['missing_games'])}試合不足（missing_games表）")
    if len(tables['extra_features']):
        problems.append('台帳に存在しない特徴量行があります（別人物・台帳欠落を確認）')
    if len(m) <= INITIAL_EXCLUDE:
        problems.append('初期1500試合を除くと採点候補がありません')
    if not missing_cols:
        numeric = f[BASE_FEATURES].apply(pd.to_numeric, errors='coerce')
        bad_text = f[BASE_FEATURES].notna() & numeric.isna()
        if bad_text.any().any():
            problems.append('特徴量に数値変換不能な値があります')
        if np.isinf(numeric.to_numpy()).any():
            problems.append('特徴量に無限大があります。由来不明のため自動補正しません')
        f[BASE_FEATURES] = numeric
        tables['missingness'] = pd.DataFrame({'feature': BASE_FEATURES,
            'missing': numeric.isna().sum().values, 'missing_rate': numeric.isna().mean().values})
    tables['classes'] = (m.assign(period=np.where(m['game_order'] <= INITIAL_EXCLUDE, '初期・採点外', '採点候補'),
                                  label=m['target_label'].map({0:'弱',1:'強'}).fillna('除外/不明'))
                           .groupby(['period', 'label']).size().rename('games').reset_index())
    tables['overview'] = pd.DataFrame([{'player': player, 'manifest_games':len(m),
        'feature_rows': len(f), 'required_history_games':len(needed),
        'missing_history_games':len(tables['missing_games']), 'same_hour_rows':len(tables['same_hour']),
        'missing_feature_columns':len(missing_cols), 'problems':len(problems)}])
    merged = None
    if not problems:
        merged = needed.merge(f[['game_id'] + BASE_FEATURES], on='game_id', how='left', validate='one_to_one')
        merged = merged.sort_values('game_order').reset_index(drop=True)
    return merged, m, tables, problems

def show_save(name, table, report_dir):
    print(f'\n{name}: {len(table)}行（全件はCSV）')
    display(table.head(30))
    table.to_csv(report_dir / (name + '.csv'), index=False, encoding='utf-8-sig')

def verify_current_audit():
    require(state.get('audit_ok', False), '先にセル4の入力監査を完了してください')
    require(state.get('signature') == config_signature(), '設定が変更されました。セル4から再検査してください')

state = {}  # 関数を変更・再実行した場合も前回結果を無効化
print('共通関数を定義しました。次はセル4の入力監査です。')


## ZIPからの入力生成：追加A1〜A5（初回だけ解析）

保存済みCSVがない場合は、セル1・2・3の後、セル4より前にこの追加セクションを実行します。

|実行順|目的|
|---|---|
|追加A1|ZIPの場所・本人名・キャッシュ先を指定|
|追加A2|元の抽出関数とキャッシュ関数を定義（解析しない）|
|追加A3|ZIP内の処理候補件数、キャッシュ、予定件数、欠けたZIPの場所を表示|
|追加A4|明示的な実行フラグをTrueにした場合のみ初回解析。全対象対局の台帳と補完前18特徴量をDriveへ保存|
|追加A5|生成ファイルを既存の検証設定へ接続。その後セル4の入力監査|

ZIPの標準保存先：
- `/content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip`
- `/content/drive/MyDrive/卒論/ultrareach.zip`

保存先は `/content/drive/MyDrive/卒論/validation_inputs/zip_cache/<player>_<fingerprint>/`。
`raw_features.csv`は補完も平滑化もしていません。`manifest.csv`には初期期間・段位17も含めます。
生成途中は250試合ごとにキャッシュし、中断後は生成済みバッチを再利用します。
同一ZIP・同一抽出関数の完全なキャッシュがある次回はA4をFalseのままで実行できます。
ZIPの中央ディレクトリは再確認しますが、XML・特徴量の再解析は行いません。

**同一時間内の順序は推測しません。** 未確定なら`game_order`を空欄とし、候補順は別列に記録します。
特徴量の生成は対局順に依存しないため先に完了できますが、時系列検証は順序が確定するまで停止します。
順序根拠が得られた場合だけ追加A1の`order_path`に確定順序CSVを指定し、根拠を記入して再接続してください。
提供ZIPが本人の初回からの全有効対局を含むかはZIPだけでは証明できません。確認フラグは実際の確認後にのみ変更します。
空・破損・段位不明のエントリは`archive_audit.csv`に残し、未解決なら完全性を確認済みにしません。


In [ ]:
# 追加A1：ZIP入力・キャッシュ設定
ZIP_INPUTS = {
    'shobon': {
        'zip_path': ROOT / 'mjlog_pf4-20_n30.zip',
        'player_name': 'しょぼんｎ',
        'full_history_confirmed': False,
        'history_evidence': '',  # 初回からの全有効対局が収録されていると確認した根拠
        'order_path': None,     # 任意：確定順序CSV(file_id, game_order)。IDのハッシュ順は不可
        'order_evidence': '',   # 上記CSVの順序を裏付ける情報源
    },
    'ultra': {
        'zip_path': ROOT / 'ultrareach.zip',
        'player_name': 'ウルトラ立直',
        'full_history_confirmed': False,
        'history_evidence': '',
        'order_path': None,
        'order_evidence': '',
    },
}
CACHE_ROOT = ROOT / 'validation_inputs' / 'zip_cache'
BATCH_SIZE = 250
zip_plan = None
generated_inputs = None
state = {}  # 入力設定変更で旧検証結果を無効化

### 追加A2：元の18特徴量抽出関数（基準コミットから無変更で収録）

このセルは定義のみです。元の欠損補完・モデル学習セルは含みません。

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import zipfile
import gzip
import gc

def extract_micro_tactical_features(zip_path, target_df, shobon_map):
    print("🚀 6,015試合の牌譜から18のミクロ戦術特徴量を一括抽出中...")
    target_files = set(target_df['file_id'].unique())
    features_list = []

    draw_prefixes = ('T', 'U', 'V', 'W')
    discard_prefixes = ('D', 'E', 'F', 'G')

    with zipfile.ZipFile(zip_path, 'r') as z:
        for idx, file_name in enumerate(target_files):
            if idx > 0 and idx % 500 == 0:
                print(f"   進捗: {idx} / {len(target_files)} 試合完了...")

            with z.open(file_name) as f:
                content = gzip.decompress(f.read())
                root = ET.fromstring(content)

                my_seat = shobon_map.get(file_name, -1)
                if my_seat == -1:
                    continue

                # --- 試合ごとのミクロ集計カウンター ---
                # 1. 守備のミクロ（一発目）
                first_discard_opp_reach_count = 0
                first_discard_raw_unsafe_count = 0
                first_discard_safe_count = 0

                # 2. 巡目別の押し引き（他家リーチ中、かつ自分がノーテン）
                opp_reach_noten_turns_early_count = 0
                opp_reach_noten_push_early_count = 0

                opp_reach_noten_turns_mid_count = 0
                opp_reach_noten_push_mid_count = 0

                opp_reach_noten_turns_late_count = 0
                opp_reach_noten_push_late_count = 0

                # 3. 点数状況別の押し引き（他家リーチ中）
                opp_reach_turns_as_last = 0
                opp_reach_push_as_last = 0

                opp_reach_turns_as_top = 0
                opp_reach_push_as_top = 0

                opp_reach_turns_as_flat = 0
                opp_reach_push_as_flat = 0

                # 4. 攻撃のミクロ（リーチ時）
                my_reach_count = 0
                my_reach_dora_sum = 0

                # 追っかけリーチ関連
                chasing_reach_count = 0
                chasing_reach_good_shape_count = 0  # 簡易判定
                chasing_reach_low_value_count = 0   # 簡易判定

                # 5. 仕掛けのミクロ（ポン・チー）
                total_rounds = 0
                rounds_with_early_meld = 0
                rounds_with_late_meld = 0
                rounds_with_meld_dora = 0

                # 6. ベタオリの技術・局終盤
                opp_reach_turns_total = 0
                opp_reach_suji_push_count = 0
                opp_reach_raw_unsafe_push_count = 0

                total_ryuukyoku_rounds = 0
                ryuukyoku_tenpai_count = 0
                ryuukyoku_noten_safe_count = 0

                current_round = None

                for child in root:
                    tag = child.tag
                    attrib = child.attrib

                    if tag == 'INIT':
                        total_rounds += 1
                        my_hai_attr = attrib.get(f'hai{my_seat}', '')
                        my_hand = [int(x) for x in my_hai_attr.split(',')] if my_hai_attr else []

                        seed = attrib.get('seed', '')
                        dora_indicators = []
                        if seed:
                            seed_list = seed.split(',')
                            if len(seed_list) > 5:
                                dora_indicators.append(int(seed_list[5]))

                        # 局開始時の点数・順位の把握
                        tens_str = attrib.get('ten', '')
                        my_rank = 1  # デフォルトトップ
                        my_score = 25000
                        if tens_str:
                            tens = [int(t) * 100 for t in tens_str.split(',')]
                            if my_seat < len(tens):
                                my_score = tens[my_seat]
                                # 順位算出
                                my_rank = 1 + sum(1 for other_score in tens if other_score > my_score)

                        discard_history = {0: [], 1: [], 2: [], 3: []}
                        active_reach_players = set()
                        shobon_draw_count = 0
                        has_melded_this_round = False
                        shobon_discarded_after_reach = False
                        reacher_seat = -1
                        is_menzen = True

                        my_reach_processed_this_round = False
                        current_round = True
                        continue

                    if not current_round:
                        continue

                    if tag == 'DORA':
                        dora_indicators.append(int(attrib.get('hai', -1)))
                        continue

                    # 他家のリーチ検知
                    if tag == 'REACH':
                        p_reach = int(attrib.get('who', -1))
                        step = str(attrib.get('step', ''))

                        if p_reach != my_seat and step == '1':
                            active_reach_players.add(p_reach)
                            reacher_seat = p_reach
                            shobon_discarded_after_reach = False
                            first_discard_opp_reach_count += 1

                        elif p_reach == my_seat and not my_reach_processed_this_round:
                            my_reach_processed_this_round = True
                            my_reach_count += 1

                            # ドラ数簡易算出
                            dora_tiles = []
                            for ind in dora_indicators:
                                ind_type = ind // 4
                                if ind_type < 27:
                                    dora_tile = ind_type + 1 if ind_type % 9 != 8 else ind_type - 8
                                else:
                                    if ind_type < 31:
                                        dora_tile = ind_type + 1 if ind_type < 30 else 27
                                    else:
                                        dora_tile = ind_type + 1 if ind_type < 33 else 31
                                dora_tiles.append(dora_tile)
                            my_dora_count = sum(1 for tile in my_hand if (tile // 4) in dora_tiles)
                            my_reach_dora_sum += my_dora_count

                            # 追いかけリーチのミクロ判定
                            if len(active_reach_players) > 0:
                                chasing_reach_count += 1
                                # 簡易良形判定（ツモ牌を含む手牌14枚に重複ペアが少ない＝多面待ちが残りやすい）
                                if len(set([t // 4 for t in my_hand])) >= 10:
                                    chasing_reach_good_shape_count += 1
                                # 簡易低打点判定（ドラが0枚）
                                if my_dora_count == 0:
                                    chasing_reach_low_value_count += 1

                    # 仕掛け（ポン・チー）の検知
                    if tag == 'N':
                        who_meld = int(attrib.get('who', -1))
                        if who_meld == my_seat:
                            is_menzen = False
                            if not has_melded_this_round:
                                has_melded_this_round = True
                                # 序盤（1〜6巡目）の仕掛け
                                if shobon_draw_count <= 6:
                                    rounds_with_early_meld += 1
                                # 終盤（13巡目以降）の仕掛け
                                elif shobon_draw_count >= 13:
                                    rounds_with_late_meld += 1

                                # ドラ持ち仕掛け
                                dora_tiles = []
                                for ind in dora_indicators:
                                    ind_type = ind // 4
                                    if ind_type < 27:
                                        dora_tile = ind_type + 1 if ind_type % 9 != 8 else ind_type - 8
                                    else:
                                        if ind_type < 31:
                                            dora_tile = ind_type + 1 if ind_type < 30 else 27
                                        else:
                                            dora_tile = ind_type + 1 if ind_type < 33 else 31
                                    dora_tiles.append(dora_tile)
                                if any((tile // 4) in dora_tiles for tile in my_hand):
                                    rounds_with_meld_dora += 1

                    # 自分がツモ
                    for seat, prefix in enumerate(draw_prefixes):
                        if tag.startswith(prefix) and tag[1:].isdigit():
                            tile_drawn = int(tag[1:])
                            if seat == my_seat:
                                my_hand.append(tile_drawn)
                                shobon_draw_count += 1

                    # 誰かが捨てる
                    for seat, prefix in enumerate(discard_prefixes):
                        if tag.startswith(prefix) and tag[1:].isdigit():
                            tile_discarded = int(tag[1:])
                            discard_history[seat].append(tile_discarded)

                            if seat == my_seat:
                                if tile_discarded in my_hand:
                                    my_hand.remove(tile_discarded)

                                # 他家リーチ中の打牌判定
                                if active_reach_players:
                                    opp_reach_turns_total += 1
                                    reacher_discards = set(discard_history[reacher_seat])
                                    is_safe = (tile_discarded // 4) in [t // 4 for t in reacher_discards]

                                    # ①一発目（初打）のミクロ判断
                                    if not shobon_discarded_after_reach:
                                        if is_safe:
                                            first_discard_safe_count += 1
                                        else:
                                            # 無筋プッシュ（リーチ宣言者の河に同じ牌がない、かつスジでもない簡易判定）
                                            # スジ判定用
                                            is_suji = False
                                            tile_type = tile_discarded // 4
                                            if tile_type < 27:
                                                num = tile_type % 9
                                                # スジ：1が安全なら4、4が安全なら1と7、9が安全なら6など
                                                # ここでは簡易的に安全牌リストに基づく無筋を検出
                                                pass
                                            first_discard_raw_unsafe_count += 1
                                        shobon_discarded_after_reach = True

                                    # 簡易ノーテン判定 (鳴いておらず、かつリーチもしていない場合をノーテンと仮定)
                                    is_noten = is_menzen and (not my_reach_processed_this_round)

                                    # ②巡目別のノーテン押し引き
                                    if is_noten:
                                        if shobon_draw_count <= 6:
                                            opp_reach_noten_turns_early_count += 1
                                            if not is_safe:
                                                opp_reach_noten_push_early_count += 1
                                        elif shobon_draw_count <= 12:
                                            opp_reach_noten_turns_mid_count += 1
                                            if not is_safe:
                                                opp_reach_noten_push_mid_count += 1
                                        else:
                                            opp_reach_noten_turns_late_count += 1
                                            if not is_safe:
                                                opp_reach_noten_push_late_count += 1

                                    # ③点数状況別の押し引き
                                    if my_rank == 4: # ラス目
                                        opp_reach_turns_as_last += 1
                                        if not is_safe:
                                            opp_reach_push_as_last += 1
                                    elif my_rank == 1: # トップ目
                                        opp_reach_turns_as_top += 1
                                        if not is_safe:
                                            opp_reach_push_as_top += 1
                                    else: # 平場（2位・3位）
                                        opp_reach_turns_as_flat += 1
                                        if not is_safe:
                                            opp_reach_push_as_flat += 1

                                    # ④ベタオリ技術（スジ押し・無筋プッシュ）
                                    if not is_safe:
                                        opp_reach_raw_unsafe_push_count += 1

                    # 局の結末が流局（RYUUKYOKU）の場合
                    if tag == 'RYUUKYOKU':
                        total_ryuukyoku_rounds += 1
                        # 自分がテンパイ流局したか（手牌開示属性から判定）
                        # 天鳳ではテンパイ者は手牌を開示するため、hai属性がログに記録される
                        my_meld_attr = child.attrib.get(f'hai{my_seat}', '')
                        if my_meld_attr:
                            ryuukyoku_tenpai_count += 1
                        else:
                            # ノーテンで放銃せずにオリきれた局数
                            # (誰かに放銃（AGARI）されずに流局を迎えられたため、ノーテンなら安全にオリきったとみなす)
                            ryuukyoku_noten_safe_count += 1

                # 1試合の解析完了。18大特徴量の算出
                features_list.append({
                    'file_id': file_name,
                    # 1. 守備のミクロ
                    'first_discard_raw_unsafe_rate': first_discard_raw_unsafe_count / first_discard_opp_reach_count if first_discard_opp_reach_count > 0 else np.nan,
                    'first_discard_safe_rate': first_discard_safe_count / first_discard_opp_reach_count if first_discard_opp_reach_count > 0 else np.nan,

                    # 2. 巡目別の押し引き
                    'early_noten_push_rate': opp_reach_noten_push_early_count / opp_reach_noten_turns_early_count if opp_reach_noten_turns_early_count > 0 else np.nan,
                    'mid_noten_push_rate': opp_reach_noten_push_mid_count / opp_reach_noten_turns_mid_count if opp_reach_noten_turns_mid_count > 0 else np.nan,
                    'late_noten_push_rate': opp_reach_noten_push_late_count / opp_reach_noten_turns_late_count if opp_reach_noten_turns_late_count > 0 else np.nan,

                    # 3. 点数状況別の押し引き
                    'las_desperate_push_rate': opp_reach_push_as_last / opp_reach_turns_as_last if opp_reach_turns_as_last > 0 else np.nan,
                    'top_conservative_push_rate': opp_reach_push_as_top / opp_reach_turns_as_top if opp_reach_turns_as_top > 0 else np.nan,
                    'flat_situation_push_rate': opp_reach_push_as_flat / opp_reach_turns_as_flat if opp_reach_turns_as_flat > 0 else np.nan,

                   # 4. 攻撃のミクロ
                        'riichi_dora_mean': my_reach_dora_sum / my_reach_count if my_reach_count > 0 else np.nan,
                        'chasing_riichi_good_shape_rate': chasing_reach_good_shape_count / chasing_reach_count if chasing_reach_count > 0 else np.nan,
                        'chasing_riichi_low_value_rate': chasing_reach_low_value_count / chasing_reach_count if chasing_reach_count > 0 else np.nan,

                        # ★★★ ここを追加（真の追っかけリーチ率） ★★★
                        'chasing_riichi_rate': chasing_reach_count / first_discard_opp_reach_count if first_discard_opp_reach_count > 0 else np.nan,
                    # 5. 仕掛けのミクロ
                    'early_meld_ratio': rounds_with_early_meld / total_rounds if total_rounds > 0 else 0.0,
                    'late_meld_ratio': rounds_with_late_meld / total_rounds if total_rounds > 0 else 0.0,
                    'meld_dora_ratio': rounds_with_meld_dora / total_rounds if total_rounds > 0 else 0.0,

                    # 6. ベタオリの技術・局終盤
                    'raw_unsafe_discard_rate': opp_reach_raw_unsafe_push_count / opp_reach_turns_total if opp_reach_turns_total > 0 else np.nan,
                    'ryuukyoku_tenpai_rate': ryuukyoku_tenpai_count / total_ryuukyoku_rounds if total_ryuukyoku_rounds > 0 else np.nan,
                    'ryuukyoku_noten_safe_rate': ryuukyoku_noten_safe_count / total_ryuukyoku_rounds if total_ryuukyoku_rounds > 0 else np.nan
                })

    return pd.DataFrame(features_list)



# 追加A2：キャッシュ・台帳生成関数（定義だけ。ZIPの本文はまだ読みません）
import os, io, contextlib, inspect, zlib
from collections import Counter
import urllib.parse

EXTRACTOR_SOURCE_COMMIT = '4d7b41dae7901b08ef7f865b8591348f2cbed642'
EXTRACTOR_SHA256 = 'f120fbf4bf1951f6fe0452a6a0aefb2bcb6ed4eae3dcb397a0485ad998e1ecc7'
CACHE_SCHEMA = 'zip-raw-v1'

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, default=str, indent=2), encoding='utf-8')
    os.replace(temporary, path)

def write_csv_atomic(path, frame):
    path = Path(path)
    temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8-sig')
    os.replace(temporary, path)

def zip_inventory(player, cfg):
    path = Path(cfg['zip_path'])
    require(path.is_file(), f'必要なZIPがありません: {path}')
    # 中央ディレクトリだけ読む。gzip展開やXML解析はしない。
    with zipfile.ZipFile(path) as archive:
        infos = [i for i in archive.infolist() if i.filename.endswith('.mjlog') and not i.is_dir()]
        require(len(infos)>0, f'.mjlogがありません: {path}')
        names = [i.filename for i in infos]
        require(len(names)==len(set(names)), 'ZIP内の同名エントリ重複があります。自動選択しません')
        items = sorted((i.filename, i.CRC, i.file_size, i.compress_size) for i in infos)
    # 軽量なキャッシュ同一性判定。ZIP本文のSHAではなく、全メンバーのCRC/サイズ/名前を使用。
    payload = {'schema': CACHE_SCHEMA, 'player':player, 'name':cfg['player_name'],
               'extractor':EXTRACTOR_SHA256, 'members':items}
    signature = hashlib.sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    return {'player':player, 'zip_path':str(path), 'signature':signature,
            'cache_dir':str(CACHE_ROOT / (player + '_' + signature[:20])),
            'archive_entries':len(items), 'member_names':[x[0] for x in items],
            'player_name':cfg['player_name']}

def valid_record(folder, marker, files, signature):
    try:
        meta = json.loads((folder / marker).read_text(encoding='utf-8'))
        return (meta['signature']==signature and meta['extractor']==EXTRACTOR_SHA256 and
                all((folder / p).is_file() and file_hash(folder / p)==meta['sha256'][p] for p in files))
    except (OSError, ValueError, KeyError):
        return False

def save_record(folder, marker, files, signature, **extra):
    write_json_atomic(folder / marker, dict(signature=signature, extractor=EXTRACTOR_SHA256,
        sha256={p:file_hash(folder / p) for p in files}, **extra))

def raw_manifest_ready(plan):
    return valid_record(Path(plan['cache_dir']), 'scan.json', ['raw_manifest.csv','archive_audit.csv'], plan['signature'])

def raw_features_ready(plan):
    return valid_record(Path(plan['cache_dir']), 'features.json', ['raw_features.csv'], plan['signature'])

def preview_zip_inputs():
    plans, rows, issues = {}, [], []
    for player,cfg in ZIP_INPUTS.items():
        try:
            plan = zip_inventory(player,cfg)
            plans[player] = plan
            folder = Path(plan['cache_dir'])
            scan_cached, features_cached = raw_manifest_ready(plan), raw_features_ready(plan)
            targets = len(pd.read_csv(folder/'raw_manifest.csv')) if scan_cached else None
            checkpoint_rows = 0
            if scan_cached and not features_cached:
                known = set(pd.read_csv(folder/'raw_manifest.csv')['file_id'])
                checkpoint_ids = set()
                for marker in sorted(folder.glob('batch_*.json')):
                    csv_name = marker.stem + '.csv'
                    if valid_record(folder, marker.name, [csv_name], plan['signature']):
                        checkpoint_ids.update(pd.read_csv(folder/csv_name)['file_id'])
                checkpoint_rows = len(checkpoint_ids & known)
            rows.append({'player':player, 'ZIP':plan['zip_path'], 'ZIP候補件数':plan['archive_entries'],
                '台帳キャッシュ':scan_cached, '本人対局数':targets,
                '特徴量キャッシュ':features_cached, '再利用可能な途中対局数':checkpoint_rows,
                'XML台帳走査予定':0 if scan_cached else plan['archive_entries'],
                '特徴量解析予定':0 if features_cached else (max(0,targets-checkpoint_rows) if targets is not None else '台帳生成後に確定'),
                '保存先':str(folder)})
        except (ValueError,OSError,zipfile.BadZipFile) as exc:
            issues.append({'player':player,'problem':str(exc)})
    display(pd.DataFrame(rows))
    if issues:
        display(pd.DataFrame(issues))
    require(not issues and len(plans)==2, 'ZIPを指定場所に置き、追加A3を再実行してください。本文解析は未実行です。')
    return plans

def scan_archive(plan):
    folder = Path(plan['cache_dir'])
    folder.mkdir(parents=True, exist_ok=True)
    if raw_manifest_ready(plan):
        print(plan['player'], '台帳キャッシュを再利用（XML再解析なし）')
        return pd.read_csv(folder/'raw_manifest.csv'), pd.read_csv(folder/'archive_audit.csv')
    rows, audit = [], []
    print(plan['player'], '台帳用XML走査件数:', plan['archive_entries'])
    with zipfile.ZipFile(plan['zip_path']) as archive:
        for number,name in enumerate(plan['member_names'],1):
            status, detail = 'ok', ''
            try:
                game_id = canonical_game_id(name)
                content = archive.read(name)
                if not content:
                    raise ValueError('ZIPエントリの内容が空')
                # 元の特徴量抽出関数と同じgzip形式を要求。別形式を黙って変換しない。
                xml = gzip.decompress(content)
                if not xml:
                    raise ValueError('gzip展開後のXMLが空')
                root = ET.fromstring(xml)
                found = []
                names_seen = set()
                for child in root:
                    if child.tag!='UN':
                        continue
                    names = [urllib.parse.unquote(child.attrib.get(f'n{i}','')) for i in range(4)]
                    names_seen.update(x for x in names if x)
                    # 誤同定を避け本人名で完全一致。18特徴量の式には変更なし。
                    seats = [i for i,x in enumerate(names) if x==plan['player_name']]
                    if len(seats)>1:
                        raise ValueError('本人名が複数席に出現')
                    if seats:
                        dan_list = child.attrib.get('dan','').split(',')
                        seat = seats[0]
                        if seat < len(dan_list) and dan_list[seat]!='':
                            found.append((seat,int(dan_list[seat])))
                if not found:
                    if plan['player_name'] in names_seen or not names_seen:
                        raise ValueError('本人の段位またはUNの名前情報が不足')
                    status, detail = 'different_player', '記録された名前: ' + ' / '.join(sorted(names_seen))
                else:
                    if len(set(found))!=1:
                        raise ValueError('UN間で本人の席順または段位が矛盾')
                    seat,dan = found[0]
                    rows.append({'file_id':name,'game_id':game_id,'dan':dan,'seat':seat,
                                 'id_hour':game_id[:10]})
            except (ValueError, OSError, EOFError, ET.ParseError, KeyError, zipfile.BadZipFile, zlib.error) as exc:
                status, detail = 'unresolved', f'{type(exc).__name__}: {exc}'
            audit.append({'file_id':name,'status':status,'detail':detail})
            if number%1000==0:
                print(f'台帳 {number}/{plan["archive_entries"]}')
    raw = pd.DataFrame(rows,columns=['file_id','game_id','dan','seat','id_hour'])
    audit = pd.DataFrame(audit,columns=['file_id','status','detail'])
    write_csv_atomic(folder/'raw_manifest.csv',raw)
    write_csv_atomic(folder/'archive_audit.csv',audit)
    save_record(folder,'scan.json',['raw_manifest.csv','archive_audit.csv'],plan['signature'])
    return raw,audit

def order_manifest(raw, cfg):
    raw = raw.copy()
    raw['id_hour'] = pd.to_datetime(raw['game_id'].str[:10],format='%Y%m%d%H',errors='coerce')
    duplicate = raw[raw['game_id'].duplicated(keep=False)]
    require(len(duplicate)==0,'正規化対局IDが重複しています。duplicates.csvを確認。自動で視点を選びません。')
    require(len(raw)>0,'本人の有効対局が0件です。名前とarchive_audit.csvを確認してください')
    ties = raw[raw['id_hour'].duplicated(keep=False) | raw['id_hour'].isna()]
    # 候補順は表示専用。未解決のgame_orderはNAとし検証セルを通さない。
    ordered = raw.sort_values(['id_hour','game_id'],kind='stable').reset_index(drop=True)
    ordered['candidate_order_NOT_VERIFIED'] = np.arange(1,len(ordered)+1)
    ordered['game_order'] = pd.Series(pd.NA,index=ordered.index,dtype='Int64')
    verified, evidence = False, ''
    if cfg['order_path'] is not None:
        path = Path(cfg['order_path'])
        require(path.is_file(),f'順序根拠CSVがありません: {path}')
        require(bool(str(cfg['order_evidence']).strip()),'順序CSVの根拠を記入してください')
        order = pd.read_csv(path)
        require({'file_id','game_order'} <= set(order.columns),'順序CSVはfile_id, game_order列が必要です')
        order['game_id'] = order['file_id'].map(canonical_game_id)
        require(order['game_id'].is_unique and set(order['game_id'])==set(raw['game_id']),
                '順序CSVの対局IDは台帳と完全一致・重複なしである必要があります')
        values = pd.to_numeric(order['game_order'],errors='coerce')
        require(np.array_equal(np.sort(values.to_numpy()),np.arange(1,len(raw)+1)),
                '順序CSVのgame_orderは1始まりの連番で指定してください')
        order['game_order'] = values.astype(int)
        ordered = ordered.drop(columns='game_order').merge(order[['game_id','game_order']],on='game_id',validate='one_to_one')
        ordered = ordered.sort_values('game_order').reset_index(drop=True)
        require(ordered['id_hour'].notna().all() and ordered['id_hour'].is_monotonic_increasing,
                '指定した順序と対局IDの日時が矛盾しています')
        verified, evidence = True, str(cfg['order_evidence']) + '; CSV SHA256=' + file_hash(path)
    elif ties.empty:
        ordered['game_order'] = np.arange(1,len(ordered)+1)
        verified, evidence = True,'全対局のID日時が有効かつ時単位で一意。提供ZIP内の順序のみを保証。'
    return ordered,ties,verified,evidence

def extract_cached_features(plan, raw):
    folder = Path(plan['cache_dir'])
    wanted = set(raw['file_id'])
    if raw_features_ready(plan):
        frame = pd.read_csv(folder/'raw_features.csv')
        require(set(frame['file_id'])==wanted and frame['file_id'].is_unique,'特徴量キャッシュの対局IDが台帳と不一致')
        require(list(frame.columns)==['file_id']+BASE_FEATURES,'特徴量キャッシュの列が元の18特徴量と不一致')
        print(plan['player'],len(frame),'試合の特徴量キャッシュ再利用（牌譜再解析なし）')
        return frame
    require(isinstance(BATCH_SIZE,int) and BATCH_SIZE>0,'BATCH_SIZEは正の整数にしてください')
    parts, completed = [],set()
    for marker in sorted(folder.glob('batch_*.json')):
        name = marker.stem+'.csv'
        if valid_record(folder,marker.name,[name],plan['signature']):
            chunk = pd.read_csv(folder/name)
            require(list(chunk.columns)==['file_id']+BASE_FEATURES and chunk['file_id'].is_unique,
                    '途中キャッシュの列・重複が不正')
            require(set(chunk['file_id'])<=wanted and not (set(chunk['file_id']) & completed),
                    '途中キャッシュの対局IDが不正または重複')
            parts.append(chunk); completed.update(chunk['file_id'])
    pending = sorted(wanted-completed)
    print(plan['player'],f'本人対局 {len(wanted)} / 再利用 {len(completed)} / 新規特徴量解析 {len(pending)}')
    seat_map = raw.set_index('file_id')['seat'].astype(int).to_dict()
    for offset in range(0,len(pending),BATCH_SIZE):
        ids = pending[offset:offset+BATCH_SIZE]
        # 元関数をそのまま呼ぶ。元の固定「6015試合」表示のみ抑止する。
        with contextlib.redirect_stdout(io.StringIO()):
            frame = extract_micro_tactical_features(plan['zip_path'],pd.DataFrame({'file_id':ids}),seat_map)
        require(set(frame['file_id'])==set(ids) and frame['file_id'].is_unique,'特徴量抽出件数が対象と一致しません')
        require(set(frame.columns)=={'file_id',*BASE_FEATURES},'元関数の出力が18特徴量ではありません')
        frame = frame[['file_id']+BASE_FEATURES].sort_values('file_id').reset_index(drop=True)
        require(not np.isinf(frame[BASE_FEATURES].to_numpy()).any(),'特徴量に無限大があります')
        batch_name = 'batch_' + hashlib.sha256('\n'.join(ids).encode()).hexdigest()[:20]
        write_csv_atomic(folder/(batch_name+'.csv'),frame)
        save_record(folder,batch_name+'.json',[batch_name+'.csv'],plan['signature'])
        parts.append(frame)
        print(plan['player'],f'特徴量保存済み {len(completed)+offset+len(ids)}/{len(wanted)}')
    require(bool(parts),'保存可能な特徴量がありません')
    result = pd.concat(parts,ignore_index=True).sort_values('file_id').reset_index(drop=True)
    require(result['file_id'].is_unique and set(result['file_id'])==wanted,'結合後の特徴量対局IDが不一致')
    write_csv_atomic(folder/'raw_features.csv',result)
    save_record(folder,'features.json',['raw_features.csv'],plan['signature'],rows=len(result))
    return result

def prepare_one_zip(plan, cfg, allow_parse):
    # プレビュー後にZIPが差し替わっていれば件数の再確認を要求。
    current = zip_inventory(plan['player'],cfg)
    require(current['signature']==plan['signature'],'ZIPがプレビュー後に変更されました。追加A3を再実行してください')
    if not (raw_manifest_ready(plan) and raw_features_ready(plan)):
        require(allow_parse,'未生成データがあります。追加A3の件数を確認後、追加A4のRUN_ZIP_PARSEをTrueにして実行してください')
    raw,audit = scan_archive(plan)
    folder = Path(plan['cache_dir'])
    duplicates = raw[raw['game_id'].duplicated(keep=False)]
    write_csv_atomic(folder/'duplicates.csv',duplicates)
    display(audit.groupby('status').size().rename('件数').reset_index())
    if len(duplicates):
        display(duplicates.head(30))
    manifest,ties,order_ok,evidence = order_manifest(raw,cfg)
    write_csv_atomic(folder/'same_hour_unresolved_candidates.csv',ties)
    write_csv_atomic(folder/'manifest.csv',manifest)
    print(plan['player'],'本人の全収録対局数:',len(raw),'同一時刻等の要確認行数:',len(ties),'順序確定:',order_ok)
    if not order_ok:
        print('順序は未確定です。特徴量は順序に依存せず生成・保存しますが、時系列検証は停止します。')
        display(ties[['file_id','id_hour']].head(30))
    features = extract_cached_features(plan,raw)
    unresolved = int(audit['status'].eq('unresolved').sum())
    full_ok = bool(cfg['full_history_confirmed'] and str(cfg['history_evidence']).strip() and unresolved==0)
    result = {'manifest_path':str(folder/'manifest.csv'),'features_path':str(folder/'raw_features.csv'),
              'dan_column':'dan','raw_features_confirmed':True,
              'features_evidence':f'元抽出関数 {EXTRACTOR_SOURCE_COMMIT}; SHA256={EXTRACTOR_SHA256}; 補完・平滑化なし',
              'full_history_confirmed':full_ok,'history_evidence':cfg['history_evidence'],
              'order_verified':order_ok,'order_evidence':evidence,
              'unresolved_archive_entries':unresolved,'raw_feature_rows':len(features),
              'manifest_sha256':file_hash(folder/'manifest.csv'),'features_sha256':file_hash(folder/'raw_features.csv')}
    write_json_atomic(folder/'connection.json',result)
    return result

## 新ルートH：順序・全履歴が未確認の観測データを用いた時間帯単位検証

**今回の実行順：セル1 → セル2 → セル3 → 追加A1 → 追加A2 → H1 → H2 → H3 → H4。**
保存済みキャッシュを直接読みます。**追加A3/A4/A5、厳密方式のセル4/5/6は、このルートでは実行しません。**
H5は学習・評価を実装した任意セルですが既定はFalseで、明示的な実行指示があるまで実行しません。
RUN_ZIP_PARSEをTrueにする必要はありません。元ZIP・CSV・キャッシュは読み取り専用です。

### 調査結果と根拠

- [天鳳公式ログ仕様](https://tenhou.net/sc/raw/)には対戦開始時刻の列、ゲーム時間、開始時刻順のソートが記載されています。
  同じIDを含む公式ログ原本が入手できれば分単位等の時刻と照合できる可能性がありますが、同値時刻や記録範囲の確認は別途必要です。
- [天鳳公式マニュアル](https://cdn.tenhou.net/man/)は牌譜・乱数種について説明していますが、確認した公開資料からはID末尾や乱数種を対局順へ変換できる根拠を確認できませんでした。
- 本ノートブックはID先頭10桁を観測時間帯として扱い、ZIP格納順・更新時刻・ID末尾・段位推移を厳密順序の証拠にはしません。
- H3では共通対局全件と同時間帯の先頭6件/人に限定してXML属性を調べます。特徴量は再抽出しません。
  時刻らしい属性があっても意味・単位・タイムゾーンを確認するまで採用しません。サンプルで見つからないことは全XMLでの不存在の証明ではありません。
- ZIP内の全候補と解析成功・除外・未解決件数をキャッシュから照合します。全収録対局を処理済みでも、生涯全対局やダウンロード元の完全性は証明できません。
  入手元台帳があればH1のREFERENCE_LISTSでID照合できます。フラグは変更しません。

### 主方式：時間帯内の並べ替えで変わらない10観測対局窓

1. 50%・2/3・5/6付近の**時間帯末尾**を境界にして3分割。ラベルで境界を調整しません。
2. 移動平均を区間ごとに作るため、評価区間の先頭9観測対局は履歴専用。
3. 採点する当該対局の時間帯が1対局だけで、10対局窓の開始が時間帯の途中でない場合だけ採用。
   例：過去時間帯の件数が3+2+4、現在時間帯が1なら合計10対局の集合は確定し、過去時間帯内の順番が未知でも平均は同じです。
4. 初期1500観測対局目の時間帯を丸ごと採点から除外。段位17は履歴に残します。
5. 共通対局は外部側に保持し、しょぼんnの補完平均から除外。共通対局を1つでも含む**学習窓全体**も除外。
   先に台帳から共通対局を削って窓を詰め直す処理はしません。

この主方式も**完全な履歴の厳密時系列評価ではありません**。窓は「観測できた10対局」であり、未収録対局がないとは保証しません。
採用される対局が限定される選択バイアスがあるため、残った件数・クラス比率・除外理由を併記します。

### 補助方式：順序の感度分析

同じ時間帯内だけでID昇順・ID降順・固定ハッシュ順の3シナリオを作成します。真の順序とは呼びません。
分割境界と補完推定母集団は共通です。学習・検証・外部の採点IDはそれぞれ3シナリオの共通集合に揃え、入力値の変化をH4で出力します。
時間帯内の未来情報が仮に混じる可能性があるため、性能値は主方式の代わりに使えません。
外部結果の良いシナリオだけを選ばず、3つを併記します。主方式とは採点母集団が違うため単純比較しません。

全検査表・未解決事項・除外ID・各窓の元対局ID・入力ハッシュは新しい`validation_reports/hourly_*`へ保存します。
元のキャッシュは上書きしません。0.8340と35.59%は従来参考値のまま保持します。


In [ ]:
# H1：保存済みキャッシュ専用の監査ルート（A4は実行しない）
# 複数のキャッシュがある場合のみ該当フォルダを明示。自動で最新版を推測しない。
HOURLY_CACHE_DIRS = {'shobon': None, 'ultra': None}
# 任意：入手元の全対局リスト等（file_id列）。比較範囲と情報源も明記する。
REFERENCE_LISTS = {'shobon': None, 'ultra': None}
REFERENCE_SCOPES = {'shobon': '', 'ultra': ''}
SCENARIOS = ['hour_invariant', 'id_ascending', 'id_descending', 'hash_seed42']
hour_state = {}

In [ ]:
# H2：関数定義（キャッシュ・元CSVは変更しない）
def hour_cache_load(player):
    cfg = ZIP_INPUTS[player]
    if HOURLY_CACHE_DIRS[player] is not None:
        folder = Path(HOURLY_CACHE_DIRS[player])
    elif Path(cfg['zip_path']).is_file():
        folder = Path(zip_inventory(player,cfg)['cache_dir'])
    else:
        choices = sorted(CACHE_ROOT.glob(player + '_*'))
        require(len(choices)==1, f'{player}: ZIP未発見または候補複数。HOURLY_CACHE_DIRSに保存済みフォルダを指定: {choices}')
        folder = choices[0]
    require((folder/'scan.json').is_file(), f'台帳キャッシュがありません: {folder}。再抽出はしません')
    signature = json.loads((folder/'scan.json').read_text(encoding='utf-8'))['signature']
    require(valid_record(folder,'scan.json',['raw_manifest.csv','archive_audit.csv'],signature), '台帳キャッシュのハッシュ不一致')
    require(valid_record(folder,'features.json',['raw_features.csv'],signature), '特徴量キャッシュが不完全またはハッシュ不一致。再抽出はしません')
    zip_match = 'ZIPなし・再照合未実施'
    if Path(cfg['zip_path']).is_file():
        require(zip_inventory(player,cfg)['signature']==signature,'ZIPと選択キャッシュが不一致')
        zip_match = '中央ディレクトリの名前/CRC/サイズ一致（本文の再解析なし）'
    m = pd.read_csv(folder/'raw_manifest.csv')
    f = pd.read_csv(folder/'raw_features.csv')
    require({'file_id','dan'}<=set(m.columns) and {'file_id',*BASE_FEATURES}<=set(f.columns),'必須列不足')
    for frame in (m,f):
        frame['game_id'] = frame['file_id'].map(canonical_game_id)
        require(frame['game_id'].is_unique,'正規化ID重複。自動削除しません')
    missing = m.loc[~m.game_id.isin(f.game_id),['file_id','game_id']]
    show_save(player+'_missing_feature_games',missing,hour_state['report_dir'])
    require(missing.empty and set(f.game_id)==set(m.game_id),'台帳と特徴量IDの不一致')
    m['dan'] = pd.to_numeric(m['dan'],errors='coerce')
    require(m['dan'].notna().all() and m['dan'].between(0,20).all() and m['dan'].mod(1).eq(0).all(),'不正段位')
    numeric = f[BASE_FEATURES].apply(pd.to_numeric,errors='coerce')
    require(not (f[BASE_FEATURES].notna() & numeric.isna()).any().any(),'非数値特徴量')
    require(not np.isinf(numeric.to_numpy()).any(),'無限大特徴量')
    f[BASE_FEATURES] = numeric
    # 既存game_order、確認フラグは書き換えず、新しい観測時刻列を使用。
    data = m.drop(columns=BASE_FEATURES,errors='ignore').merge(f[['game_id']+BASE_FEATURES],on='game_id',validate='one_to_one')
    data['hour'] = pd.to_datetime(data['game_id'].str[:10],format='%Y%m%d%H',errors='coerce')
    require(data['hour'].notna().all(),'IDの時間帯が解釈不能')
    data['target_label'] = np.select([data.dan>=18,data.dan<=16],[1,0],default=np.nan)
    data = data.sort_values(['hour','game_id']).reset_index(drop=True)
    require(len(data)>INITIAL_EXCLUDE,'初期1500観測対局除外後のデータが不足')
    cutoff = data.iloc[INITIAL_EXCLUDE-1]['hour']
    # 初期1500件目と同じ時間帯を丸ごと採点外にする。完全な生涯順序とは主張しない。
    audit = pd.read_csv(folder/'archive_audit.csv')
    require(set(audit.loc[audit.status.eq('ok'),'file_id'])==set(m.file_id),
            'XML走査成功一覧と台帳が不一致')
    show_save(player+'_archive_status',audit.groupby('status').size().rename('entries').reset_index(),hour_state['report_dir'])
    show_save(player+'_unresolved_archive_entries',audit[audit.status.eq('unresolved')],hour_state['report_dir'])
    show_save(player+'_hour_sizes',data.groupby('hour').size().rename('games').reset_index(),hour_state['report_dir'])
    show_save(player+'_missing_values',numeric.isna().sum().rename('missing').rename_axis('feature').reset_index(),hour_state['report_dir'])
    gaps = data[['hour']].drop_duplicates().sort_values('hour')
    gaps['gap_hours'] = gaps.hour.diff().dt.total_seconds()/3600
    show_save(player+'_observed_gaps_NOT_PROOF_OF_MISSING',gaps.sort_values('gap_hours',ascending=False).head(20),hour_state['report_dir'])
    reference_status = '外部の完全な台帳なし・生涯履歴完全性は未確認'
    if REFERENCE_LISTS[player] is not None:
        require(bool(REFERENCE_SCOPES[player].strip()),'参照台帳の情報源と範囲を記入してください')
        ref = pd.read_csv(REFERENCE_LISTS[player])
        require('file_id' in ref,'参照台帳にfile_idが必要')
        ref['game_id'] = ref.file_id.map(canonical_game_id)
        require(ref.game_id.is_unique,'参照台帳に対局ID重複')
        absent = ref[~ref.game_id.isin(data.game_id)]
        extra = data[~data.game_id.isin(ref.game_id)][['file_id','game_id']]
        show_save(player+'_reference_missing_games',absent,hour_state['report_dir'])
        show_save(player+'_outside_reference_scope',extra,hour_state['report_dir'])
        reference_status = f'指定参照範囲との比較のみ: 欠落{len(absent)} / 範囲外{len(extra)}; {REFERENCE_SCOPES[player]}'
    connection = json.loads((folder/'connection.json').read_text(encoding='utf-8')) if (folder/'connection.json').exists() else {}
    facts = dict(player=player,rows=len(data),cutoff_hour=str(cutoff),
                 initial_excluded=int(data.hour.le(cutoff).sum()),
                 order_verified_original=connection.get('order_verified',False),
                 full_history_confirmed_original=connection.get('full_history_confirmed',False),
                 zip_check=zip_match,history_status=reference_status,
                 unresolved_entries=int(audit.status.eq('unresolved').sum()),
                 raw_manifest_sha256=file_hash(folder/'raw_manifest.csv'),
                 raw_features_sha256=file_hash(folder/'raw_features.csv'),cache_dir=str(folder))
    return data,cutoff,facts

def xml_evidence_sample(data_by_player,shared,report_dir):
    # 共通対局全件＋同一時間帯の先頭6件/人だけを確認。特徴量抽出関数は呼ばない。
    rows = []
    for player,data in data_by_player.items():
        tied = data[data.hour.duplicated(keep=False)].head(6)
        subset = pd.concat([data[data.game_id.isin(shared)],tied]).drop_duplicates('game_id')
        path = Path(ZIP_INPUTS[player]['zip_path'])
        print(player,'XMLメタデータ確認予定:',len(subset),'件（特徴量の再抽出なし）')
        if not path.is_file():
            rows.append(dict(player=player,game_id='',status='ZIP未発見・XML証拠未確認',detail=str(path)))
            continue
        with zipfile.ZipFile(path) as archive:
            for row in subset.itertuples():
                record=dict(player=player,game_id=row.game_id,shared=row.game_id in shared)
                try:
                    xml = gzip.decompress(archive.read(row.file_id))
                    root = ET.fromstring(xml)
                    names=set()
                    fields=[]
                    for node in root.iter():
                        if node.tag=='UN':
                            names.update(urllib.parse.unquote(node.attrib.get(f'n{i}','')) for i in range(4))
                        for key,value in node.attrib.items():
                            if re.search(r'time|date|stamp',key,re.I):
                                fields.append({'tag':node.tag,'attribute':key,'raw_value':value[:200]})
                    record.update(status='確認済み（順序への変換なし）',
                        contains_shobon=ZIP_INPUTS['shobon']['player_name'] in names,
                        contains_ultra=ZIP_INPUTS['ultra']['player_name'] in names,
                        xml_sha256=hashlib.sha256(xml).hexdigest(),
                        time_named_attributes=json.dumps(fields,ensure_ascii=False),
                        detail='属性名候補のみ。時刻の意味・単位は未検証。未検出でも全XMLの不存在を証明しない。')
                except (ValueError,OSError,EOFError,ET.ParseError,KeyError) as exc:
                    record.update(status='未確認',detail=str(exc))
                rows.append(record)
    result=pd.DataFrame(rows)
    show_save('xml_time_and_shared_game_evidence',result,report_dir)
    return result

def arrange_hours(data,scenario):
    result=data.copy()
    if scenario=='hash_seed42':
        result['_tie']=result.game_id.map(lambda x:hashlib.sha256(('42:'+x).encode()).hexdigest())
    else:
        result['_tie']=result.game_id
    result=result.sort_values(['hour','_tie'],ascending=[True,scenario!='id_descending']).drop(columns='_tie').reset_index(drop=True)
    return result

def hour_means(rows,cutoff,forbidden):
    fitting=rows[(rows.hour>cutoff)&~rows.game_id.isin(forbidden)]
    means=fit_imputation(fitting)
    return means,set(fitting.game_id)

def prepare_hour_block(rows,means,cutoff,scenario,forbidden=frozenset()):
    require(len(rows)>=WINDOW,'区間に10対局未満しかありません')
    require(rows.game_id.is_unique,'区間の対局ID重複')
    ordered=arrange_hours(rows,scenario)
    hour_sizes=ordered.groupby('hour').size()
    filled=ordered[BASE_FEATURES].fillna(means)
    require(np.isfinite(filled.to_numpy()).all(),'補完後に非有限値')
    rolled=add_cross(filled).rolling(WINDOW,min_periods=WINDOW).mean()
    ids=ordered.game_id.tolist();hours=ordered.hour.tolist()
    metadata=[];keep=[];excluded=[]
    for i,row in ordered.iterrows():
        start=i-WINDOW+1
        source=tuple(ids[max(0,start):i+1])
        reason=''
        if start<0:reason='区間先頭9対局・履歴専用'
        elif row.hour<=cutoff:reason='観測初期1500対局と同時間帯・採点外'
        elif pd.isna(row.target_label):reason='段位17・履歴のみ'
        elif set(source)&set(forbidden):reason='外部共通対局を含む学習窓'
        elif scenario=='hour_invariant':
            if hour_sizes.loc[row.hour]!=1:reason='当該対局の時間帯内順序が未確定'
            elif start>0 and hours[start-1]==hours[start]:reason='窓の開始位置が時間帯の途中'
        if reason:
            excluded.append({'game_id':row.game_id,'reason':reason})
        else:
            keep.append(i)
            metadata.append({'game_id':row.game_id,'hour':row.hour,'dan':row.dan,
                             'target_label':int(row.target_label),'source_ids':source,
                             'source_hours':tuple(str(x) for x in hours[start:i+1])})
    meta=pd.DataFrame(metadata,columns=['game_id','hour','dan','target_label','source_ids','source_hours'])
    return dict(X=rolled.iloc[keep].reset_index(drop=True),y=meta.target_label.astype(int),meta=meta,
                excluded=pd.DataFrame(excluded,columns=['game_id','reason']))

def hour_splits(data,cutoff):
    active=data[data.hour>cutoff].sort_values(['hour','game_id'])
    require(len(active)>=60,'観測初期期間後が60対局未満')
    boundaries=[active.iloc[max(0,int(len(active)*q)-1)].hour for q in (.5,2/3,5/6)]
    boundaries.append(active.hour.max())
    require(len(set(boundaries))==4,'時間帯境界が重複し3分割不可')
    return [(k+1,data[data.hour<=a],data[(data.hour>a)&(data.hour<=b)])
            for k,(a,b) in enumerate(zip(boundaries[:-1],boundaries[1:]))]

def save_prepared_meta(name,prepared,folder):
    prepared['meta'].to_csv(folder/(name+'_windows.csv'),index=False)
    prepared['excluded'].to_csv(folder/(name+'_excluded.csv'),index=False)

def intersect_sensitivity(blocks):
    # 同じ採点IDに揃えてから比較。スコアやラベルでシナリオを選ばない。
    common=set.intersection(*(set(b['meta'].game_id) for b in blocks.values()))
    ordered_ids=sorted(common)
    result={}
    for scenario,b in blocks.items():
        positions=b['meta'].reset_index().set_index('game_id').loc[ordered_ids,'index'].to_numpy(dtype=int)
        result[scenario]=dict(X=b['X'].iloc[positions].reset_index(drop=True),
            y=b['y'].iloc[positions].reset_index(drop=True),meta=b['meta'].iloc[positions].reset_index(drop=True),excluded=b['excluded'])
    return result

In [ ]:
# H3：キャッシュ読込・履歴監査・共通対局確認
hour_state={}
folder=REPORT_ROOT/('hourly_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
folder.mkdir(parents=True,exist_ok=False)
hour_state['report_dir']=folder
data_by_player={};cutoffs={};facts=[]
for p in ('shobon','ultra'):
    data_by_player[p],cutoffs[p],fact=hour_cache_load(p)
    facts.append(fact)
shared=set(data_by_player['shobon'].game_id)&set(data_by_player['ultra'].game_id)
common=data_by_player['shobon'][['game_id','file_id','dan','hour']].merge(
    data_by_player['ultra'][['game_id','file_id','dan']],on='game_id',suffixes=('_shobon','_ultra'),validate='one_to_one')
show_save('common_games_EXCLUDE_FROM_TRAINING_SOURCES',common,folder)
show_save('history_and_cache_audit',pd.DataFrame(facts),folder)
evidence=xml_evidence_sample(data_by_player,shared,folder)
shared_check=[]
for gid in sorted(shared):
    subset=evidence[evidence.game_id.eq(gid)]
    verified=(len(subset)==2 and subset.get('contains_shobon',pd.Series(False,index=subset.index)).fillna(False).all()
              and subset.get('contains_ultra',pd.Series(False,index=subset.index)).fillna(False).all())
    digests=subset.get('xml_sha256',pd.Series(dtype=str)).dropna()
    shared_check.append(dict(game_id=gid,both_players_confirmed_in_both_copies=verified,
        decoded_xml_identical=len(digests)==2 and digests.nunique()==1,
        action='学習窓・補完推定から除外。XMLが未確認でも共通IDとして保守的に除外'))
show_save('shared_game_identity_review',pd.DataFrame(shared_check),folder)
REFERENCE_RESULTS.to_csv(folder/'historical_reference_only.csv',index=False)
limits=pd.DataFrame([
    ['evaluation_scope','観測済み対局内の時間帯単位評価。厳密な全履歴時系列評価ではない'],
    ['order','order_verifiedは変更しない。主方式は順序不変の10観測対局窓のみ'],
    ['history','full_history_confirmedは変更しない。ZIPから欠落した未知の対局は検出保証不可'],
    ['initial_period','観測台帳の先頭1500対局と同時間帯を採点外。生涯初期1500対局と同一とは未確認'],
    ['shared_games',f'{len(shared)}共通対局IDを学習窓と補完推定から除外。外部側は保持'],
    ['sensitivity','ID昇順/降順/固定ハッシュ順は真の時系列ではない。時間帯内未来参照の可能性があり補助分析専用'],
    ['sampling','主方式は同時間帯複数対局や境界窓を除外するため選択バイアスの可能性'],
    ['external','ウルトラ立直は外部評価のみ。調整・シナリオ選択・閾値最適化に使用しない'],
    ['historical_overlap','外部の対局年代が訓練年代より前を含み得る。未知人物への後ろ向き検証であり将来予測ではない'],
    ['xml_evidence','共通対局全件と同時間帯サンプルだけ確認。時刻らしい属性を自動変換・確定しない'],
    ['official_source','https://tenhou.net/sc/raw/ : 開始時刻列と開始時刻順を公式説明。原本が入手できればID照合による追加検証が可能'],
],columns=['topic','limitation'])
show_save('assumptions_and_limitations',limits,folder)
hour_state.update(data=data_by_player,cutoffs=cutoffs,shared=shared,facts=facts,audit_ok=True)
write_json_atomic(folder/'hourly_configuration.json',dict(scenarios=SCENARIOS,window=WINDOW,initial_exclude=INITIAL_EXCLUDE,
    model_plan=MODEL_PLAN,threshold=THRESHOLD_PLAN,cache_facts=facts,
    source_commit='9ba0c51620002a6adfa1027f7a2db1eb9d2d51e6'))
print('時間帯方式の監査完了。厳密方式の確認フラグは変更していません。保存先:',folder)

In [ ]:
# H4：時間帯単位の分割・順序不変窓・感度分析入力を作成（モデル学習なし）
require(hour_state.get('audit_ok',False),'H3を先に実行してください')
hour_state.pop('prepared',None)
prepared={};summaries=[];failures=[]
folder=hour_state['report_dir'];d=hour_state['data'];cutoffs=hour_state['cutoffs'];shared=hour_state['shared']
for scenario in SCENARIOS:
    folds=[]
    for k,tr,va in hour_splits(d['shobon'],cutoffs['shobon']):
        row=dict(scenario=scenario,fold=k,train_last_hour=str(tr.hour.max()),validation_first_hour=str(va.hour.min()))
        try:
            require(tr.hour.max()<va.hour.min(),'分割が時間帯内にまたがっています')
            means,fit_ids=hour_means(tr,cutoffs['shobon'],shared)
            train=prepare_hour_block(tr,means,cutoffs['shobon'],scenario,shared)
            validation=prepare_hour_block(va,means,cutoffs['shobon'],scenario)
            overlap=source_ids(train)&source_ids(validation)
            require(not overlap and not (fit_ids&source_ids(validation)),'学習/評価の元対局ID重複')
            require(not (source_ids(train)&set(d['ultra'].game_id)) and not (fit_ids&set(d['ultra'].game_id)),
                    '外部対局が学習または補完推定に混入')
            row.update(train_weak=int(train['y'].eq(0).sum()),train_strong=int(train['y'].eq(1).sum()),
                val_weak=int(validation['y'].eq(0).sum()),val_strong=int(validation['y'].eq(1).sum()),
                shared_window_exclusions=int(train['excluded'].reason.eq('外部共通対局を含む学習窓').sum()),
                source_overlap=len(overlap),auc_available=validation['y'].nunique()==2)
            require(train['y'].nunique()==2,'学習側に両クラスがありません')
            require(len(validation['X'])>0,'評価候補が0件です')
            means.to_csv(folder/f'{scenario}_fold{k}_training_only_means.csv')
            save_prepared_meta(f'{scenario}_fold{k}_train',train,folder)
            save_prepared_meta(f'{scenario}_fold{k}_validation',validation,folder)
            folds.append(dict(fold=k,train=train,validation=validation,means=means))
            row['status']='OK' if validation['y'].nunique()==2 else 'OK・単一クラスAUC不可'
        except ValidationBlocked as exc:
            row['status']='STOP: '+str(exc);failures.append(row.copy())
        summaries.append(row)
    means,fit_ids=hour_means(d['shobon'],cutoffs['shobon'],shared)
    train=prepare_hour_block(d['shobon'],means,cutoffs['shobon'],scenario,shared)
    external=prepare_hour_block(d['ultra'],means,cutoffs['ultra'],scenario)
    require(not (source_ids(train)&source_ids(external)) and not (fit_ids&set(d['ultra'].game_id)),
            '外部対局が最終学習の入力・補完に混入')
    require(train['y'].nunique()==2 and len(external['X'])>0,'最終学習側の両クラスまたは外部候補が不足')
    save_prepared_meta(scenario+'_final_train',train,folder)
    save_prepared_meta(scenario+'_external',external,folder)
    prepared[scenario]=dict(folds=folds,train=train,external=external,means=means)
show_save('hourly_fold_summary',pd.DataFrame(summaries),folder)
require(not failures,'分割条件を満たさない場合があります。境界を勝手に変えず停止します')
# 補助3シナリオは同じ対局IDで比較。主方式は別母集団なので混ぜない。
sens=[s for s in SCENARIOS if s!='hour_invariant']
for k in range(3):
    aligned_train=intersect_sensitivity({s:prepared[s]['folds'][k]['train'] for s in sens})
    for s,b in aligned_train.items(): prepared[s]['folds'][k]['train']=b
    aligned=intersect_sensitivity({s:prepared[s]['folds'][k]['validation'] for s in sens})
    for s,b in aligned.items(): prepared[s]['folds'][k]['validation']=b
aligned=intersect_sensitivity({s:prepared[s]['external'] for s in sens})
for s,b in aligned.items():prepared[s]['external']=b
aligned_train=intersect_sensitivity({s:prepared[s]['train'] for s in sens})
for s,b in aligned_train.items():prepared[s]['train']=b
require(all(len(prepared[s]['external']['X'])>0 and all(len(f['validation']['X'])>0 for f in prepared[s]['folds']) for s in SCENARIOS),
        '感度分析の共通採点対象が0件です')
require(all(prepared[s]['train']['y'].nunique()==2 and all(f['train']['y'].nunique()==2 for f in prepared[s]['folds']) for s in SCENARIOS),
        '共通IDへ揃えた後の学習側に両クラスがありません')
counts=[]
for s,b in prepared.items():
    for label,v in [('final_train',b['train']),('external',b['external'])]+[(f'fold{f["fold"]}_{part}',f[part]) for f in b['folds'] for part in ('train','validation')]:
        counts.append(dict(scenario=s,dataset=label,rows=len(v['X']),weak=int(v['y'].eq(0).sum()),strong=int(v['y'].eq(1).sum())))
        save_prepared_meta(s+'_aligned_'+label,v,folder)
show_save('final_scoring_populations',pd.DataFrame(counts),folder)
drift=[]
base=prepared[sens[0]]['external']['X']
for s in sens[1:]:
    delta=(prepared[s]['external']['X']-base).abs()
    for feature in FEATURES:
        drift.append(dict(comparison=s+' minus '+sens[0],feature=feature,
                          mean_abs_change=delta[feature].mean(),max_abs_change=delta[feature].max()))
show_save('external_input_order_sensitivity_NOT_PERFORMANCE',pd.DataFrame(drift),folder)
hour_state['prepared']=prepared
print('入力準備完了。順序不変窓の主分析と、仮順序の補助分析を分離しました。学習未実行。')

In [ ]:
# H5：任意の評価実行（今回は実行しない。明示指示後のみTrueにする）
RUN_MODEL_EVALUATION = False
if not RUN_MODEL_EVALUATION:
    print('学習・外部性能評価は実行していません。H4までの監査結果を確認してください。')
else:
    require('prepared' in hour_state,'H4を先に完了してください')
    import lightgbm as lgb
    from sklearn.metrics import roc_auc_score,accuracy_score,balanced_accuracy_score,f1_score,confusion_matrix
    # 再実行で既存結果を上書きしない。外部の結果で設定/閾値/シナリオを選ばない。
    result_dir=hour_state['report_dir']/('model_'+uuid.uuid4().hex[:8]);result_dir.mkdir()
    metrics=[]
    for scenario,pack in hour_state['prepared'].items():
        tasks=[(f'fold{f["fold"]}',f['train'],f['validation']) for f in pack['folds']]
        tasks.append(('external',pack['train'],pack['external']))
        for name,train,test in tasks:
            require(train['y'].nunique()==2 and len(test['X'])>0,'学習/評価対象が不正')
            require(not (source_ids(train)&source_ids(test)),'学習/評価窓が重複')
            model=lgb.LGBMClassifier(**MODEL_PLAN)
            # eval_set/early stoppingなし。fitはしょぼんn由来の行列のみ。
            model.fit(train['X'],train['y'])
            probabilities=model.predict_proba(test['X'])[:,1]
            predictions=(probabilities>=THRESHOLD_PLAN).astype(int)
            truth=test['y'];both=truth.nunique()==2
            cm=confusion_matrix(truth,predictions,labels=[0,1])
            metrics.append(dict(scenario=scenario,dataset=name,n=len(truth),strong_ratio=truth.mean(),
                roc_auc=roc_auc_score(truth,probabilities) if both else np.nan,
                accuracy=accuracy_score(truth,predictions),
                balanced_accuracy=balanced_accuracy_score(truth,predictions) if both else np.nan,
                f1_strong=f1_score(truth,predictions,pos_label=1,zero_division=0),
                f1_macro=f1_score(truth,predictions,labels=[0,1],average='macro',zero_division=0),
                all_weak_accuracy=1-truth.mean(),all_strong_accuracy=truth.mean(),
                tn=int(cm[0,0]),fp=int(cm[0,1]),fn=int(cm[1,0]),tp=int(cm[1,1])))
            out=test['meta'][['game_id','hour','target_label']].copy()
            out['strong_score']=probabilities;out['predicted_label']=predictions
            out.to_csv(result_dir/f'{scenario}_{name}_predictions.csv',index=False)
    show_save('metrics_NOT_STRICT_CHRONOLOGY',pd.DataFrame(metrics),result_dir)
    write_json_atomic(result_dir/'model_settings.json',dict(parameters=MODEL_PLAN,threshold=THRESHOLD_PLAN,
                     lightgbm=lgb.__version__,scenarios=SCENARIOS,external_tuning=False))
    print('主方式と補助方式を区別して解釈してください。外部スコアで最良シナリオを選択しないでください。')

In [ ]:
# 追加A3：実行前プレビュー（ZIP中央ディレクトリと小さなキャッシュだけを確認）
state = {}
zip_plan = None
generated_inputs = None
zip_plan = preview_zip_inputs()
print('まだZIP本文の大量解析・モデル学習は実行していません。')
print('同じZIPと元関数のキャッシュは再利用します。キャッシュはZIP内CRC・サイズ・名前と元関数のSHAで識別します。')

In [ ]:
# 追加A4：台帳と18特徴量の生成・保存（明示的に実行するセル）
RUN_ZIP_PARSE = False  # 初回のみ、A3の件数を確認してからTrueへ。大量解析を開始します。
# 次回はFalseのままで完全なキャッシュを再利用できます。
state = {}
generated_inputs = None
require(zip_plan is not None,'追加A3を先に実行してください')
require(all(zip_inventory(p,ZIP_INPUTS[p])['signature']==plan['signature'] for p,plan in zip_plan.items()),
        'ZIP設定または内容が変わりました。追加A3から再実行してください')
needs_parse = [p for p,plan in zip_plan.items() if not (raw_manifest_ready(plan) and raw_features_ready(plan))]
require(RUN_ZIP_PARSE or not needs_parse,
        '新規解析が必要なプレイヤー: '+str(needs_parse)+ '。A3の件数を確認してRUN_ZIP_PARSE=Trueにしてください。')
results, failures = {},[]
for player,plan in zip_plan.items():
    try:
        results[player] = prepare_one_zip(plan,ZIP_INPUTS[player],RUN_ZIP_PARSE)
    except (ValueError,OSError,ET.ParseError,EOFError,zipfile.BadZipFile) as exc:
        failures.append({'player':player,'problem':str(exc),'cache_dir':plan['cache_dir']})
if failures:
    display(pd.DataFrame(failures))
require(not failures and len(results)==2,'入力生成で停止。成功したキャッシュ・バッチは保持され、次回再利用します。')
generated_inputs = results
display(pd.DataFrame([{'player':p,'特徴量件数':r['raw_feature_rows'],'順序確定':r['order_verified'],
                      '履歴完全性確認済み':r['full_history_confirmed'],'未解決ZIPエントリ':r['unresolved_archive_entries'],
                      '台帳':r['manifest_path'],'特徴量':r['features_path']} for p,r in results.items()]))
print('生成・保存完了。補完・移動平均・学習は未実行です。次は追加A5で既存検証セルに接続します。')

In [ ]:
# 追加A5：検証ノートブックへ接続（未確定情報をTrueに変更しない）
require(generated_inputs is not None,'追加A4を完了してください（キャッシュ再利用でも可）')
state = {}  # 古い検査行列を無効化
MEMORY_INPUTS = {}  # 生成ファイルを使用。旧ランタイムのDataFrameは混ぜない。
for player,record in generated_inputs.items():
    require(file_hash(record['manifest_path'])==record['manifest_sha256'] and
            file_hash(record['features_path'])==record['features_sha256'],
            '生成後に入力ファイルが変更されました。追加A4から再確認してください')
    SOURCES[player] = {k:record[k] for k in (
        'manifest_path','features_path','dan_column','raw_features_confirmed','full_history_confirmed',
        'order_verified','order_evidence','history_evidence','features_evidence')}
display(pd.DataFrame([{'player':p,**cfg} for p,cfg in SOURCES.items()]))
print('接続しました。次はセル4「入力監査」です。順序・完全性が未確認なら、その理由を表示して停止します。')

## セル4：入力監査・不足対局一覧（実データ読込、学習なし）

両プレイヤーを独立に検査してから停止判定します。未発見のファイルは勝手に探索しません。
manifest自体がない場合、必要試合数は「未確定」です。完全な台帳がある場合に限り不足対局を全件CSV出力します。
別フォルダ・別視点の同一対局も外部分離違反として停止します。自動的な除外は行いません。


In [ ]:
state = {'audit_ok': False, 'folds_ok': False, 'signature': config_signature()}
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
report_dir = REPORT_ROOT / run_id
report_dir.mkdir(parents=True, exist_ok=False)
state['report_dir'] = report_dir
prepared, manifests, errors, origins = {}, {}, [], []
for player, cfg in SOURCES.items():
    loaded = {}
    for kind in ('manifest', 'features'):
        try:
            loaded[kind], origin = load_table(player, kind)
            origins.append(origin)
        except (ValueError, OSError, ImportError, KeyError) as exc:
            errors.append({'player':player, 'problem':f'{kind}: {exc}'})
    if len(loaded) != 2:
        continue
    try:
        data, manifest, tables, problems = audit_player(player, loaded['manifest'], loaded['features'], cfg)
        manifests[player] = manifest
        for name, table in tables.items():
            show_save(player + '_' + name, table, report_dir)
        errors.extend({'player':player, 'problem':p} for p in problems)
        if data is not None:
            prepared[player] = data
    except (ValueError, KeyError) as exc:
        errors.append({'player':player, 'problem':str(exc)})

if len(manifests) == 2:
    common = manifests['shobon'][['game_id', 'file_id']].merge(
        manifests['ultra'][['game_id', 'file_id']], on='game_id', suffixes=('_shobon','_ultra'))
    show_save('cross_player_shared_games', common, report_dir)
    if len(common):
        errors.append({'player':'both', 'problem':f'プレイヤー間に同一対局{len(common)}行。分離方針の決定が必要です'})
show_save('input_provenance', pd.DataFrame(origins), report_dir)
show_save('blocking_issues', pd.DataFrame(errors, columns=['player','problem']), report_dir)
REFERENCE_RESULTS.to_csv(report_dir / 'historical_reference_only.csv', index=False)
(report_dir / 'config.json').write_text(json.dumps({
    'sources':SOURCES, 'window':WINDOW, 'initial_exclude':INITIAL_EXCLUDE,
    'model_plan_only':MODEL_PLAN, 'threshold_plan':THRESHOLD_PLAN,
    'pandas':pd.__version__, 'numpy':np.__version__, 'baseline_commit':'4d7b41dae7901b08ef7f865b8591348f2cbed642',
}, default=str, ensure_ascii=False, indent=2), encoding='utf-8')
print('検査表の保存先:', report_dir)
require(not errors and len(prepared) == 2,
        '入力監査で停止。blocking_issues.csvとmissing_games.csvを確認してください。学習・牌譜解析はしていません。')
state.update(audit_ok=True, data=prepared, origins=origins)
print('入力監査合格。次はセル5です。')


## セル5：時系列3分割・共通前処理・元対局ID監査

境界は初期期間後の対局数だけで固定し、段位やスコアで調整しません。
各評価区間の先頭9試合は履歴専用。窓は区間ごとにリセットし、学習との重複をゼロにします。
欠損補完平均はその分割の**学習区間の1501試合目以降（段位17も含む）**だけから算出します。
初期期間の末尾9試合は履歴専用で、平均の推定・学習ラベルには使いません。
評価側が単一クラスの場合はAUC不可と記録します。学習側に両クラスがなければ停止します。
各foldは独立した評価です。過去foldの評価期間が後のfoldの学習期間に入るのは拡大型分割の仕様です。


In [ ]:
state['folds_ok'] = False
state.pop('folds', None)
state.pop('external_ready', None)
verify_current_audit()
folds, fold_rows, fold_errors = [], [], []
for fold_no, train_rows, val_rows in make_folds(state['data']['shobon']):
    record = {'fold':fold_no, 'train_end_order':int(train_rows['game_order'].max()),
              'validation_start_order':int(val_rows['game_order'].min()),
              'validation_end_order':int(val_rows['game_order'].max()),
              'validation_history_only':min(WINDOW-1, len(val_rows))}
    try:
        fitting_rows = train_rows[train_rows['game_order'] > INITIAL_EXCLUDE]
        means = fit_imputation(fitting_rows)
        train = transform_block(train_rows, means)
        validation = transform_block(val_rows, means)
        require(len(train['X']) > 0 and len(validation['X']) > 0, '学習または評価の採点対象が0件です')
        shared = source_ids(train) & source_ids(validation)
        fit_shared = set(fitting_rows['game_id']) & source_ids(validation)
        record.update(train_weak=int(train['y'].eq(0).sum()), train_strong=int(train['y'].eq(1).sum()),
                      validation_weak=int(validation['y'].eq(0).sum()), validation_strong=int(validation['y'].eq(1).sum()),
                      window_overlap=len(shared), imputation_fit_overlap=len(fit_shared),
                      train_both_classes=train['y'].nunique()==2,
                      auc_available=validation['y'].nunique()==2)
        require(not shared and not fit_shared, '元対局IDまたは補完推定データが評価側と重複しています')
        require(train['y'].nunique() == 2, '学習側が単一クラスです。境界を勝手に変更せず停止します')
        means.rename('mean').to_csv(report_dir / f'fold{fold_no}_imputation_means.csv')
        for split_name, block in [('train',train),('validation',validation)]:
            block['meta'].to_csv(report_dir / f'fold{fold_no}_{split_name}_windows.csv', index=False)
        folds.append({'fold':fold_no, 'means':means, 'train':train, 'validation':validation})
        record['status'] = 'OK' if record['auc_available'] else 'OK（評価側単一クラス・AUC不可）'
    except ValidationBlocked as exc:
        record['status'] = 'STOP: ' + str(exc)
        fold_errors.append(str(exc))
    fold_rows.append(record)
show_save('fold_summary', pd.DataFrame(fold_rows), report_dir)
require(not fold_errors and len(folds)==3, '分割監査で停止。学習は実行していません。')
state.update(folds_ok=True, folds=folds)
print('3分割の入力行列をstate["folds"]に作成しました。モデルは未学習です。')


## セル6：外部予測用の入力準備・監査（予測・性能評価なし）

しょぼんn全学習期間からだけ補完平均を算出し、同じ関数で両プレイヤーを変換します。
ウルトラ立直の特徴量・段位を使って平均、モデル、閾値を調整しません。
両人物の初期1500試合・段位17は採点対象外ですが、窓の履歴には残ります。
外部予測値は生成しません。ここまで合格してから、別の指示で学習・評価を実装します。


In [ ]:
state.pop('external_ready', None)
verify_current_audit()
require(state.get('folds_ok', False), 'セル5の3分割監査を先に完了してください')
shobon = state['data']['shobon']
ultra = state['data']['ultra']
final_means = fit_imputation(shobon[shobon['game_order'] > INITIAL_EXCLUDE])
final_train = transform_block(shobon, final_means)
external_input = transform_block(ultra, final_means)
require(final_train['y'].nunique()==2, '最終学習用の入力に両クラスがありません')
require(len(external_input['X']) > 0, '外部の採点候補がありません')
shared = source_ids(final_train) & source_ids(external_input)
require(not shared, '外部用の窓に学習側と同一対局があります')
summary = pd.DataFrame([
    {'dataset':name, 'rows':len(block['X']), 'features':block['X'].shape[1],
     'weak':int(block['y'].eq(0).sum()), 'strong':int(block['y'].eq(1).sum()),
     'strong_ratio':float(block['y'].mean()), 'remaining_na':int(block['X'].isna().sum().sum()),
     'both_classes':block['y'].nunique()==2}
    for name,block in [('shobon_final_training_input',final_train), ('ultra_external_input',external_input)]
])
show_save('external_input_summary_NO_PERFORMANCE', summary, report_dir)
final_means.rename('mean').to_csv(report_dir / 'final_training_only_imputation_means.csv')
for name, block in [('shobon_final',final_train), ('ultra_external',external_input)]:
    block['meta'].to_csv(report_dir / (name + '_windows.csv'), index=False)
    pd.concat([block['meta'][['game_id','game_order']],block['X'],block['y'].rename('target_label')],axis=1).to_csv(
        report_dir / (name + '_prepared_input.csv'), index=False)
(report_dir / 'feature_order.json').write_text(json.dumps(FEATURES, indent=2), encoding='utf-8')
state['external_ready'] = {'means':final_means, 'train':final_train, 'external':external_input}
print('入力準備・監査完了。モデル学習・外部予測・指標計算は未実行です。')
print('保存先:', report_dir)
if external_input['y'].nunique() != 2:
    print('注意：外部採点対象が単一クラスのため、今後ROC-AUCと2クラスのBalanced Accuracyは評価できません。')


## 結果の読み方と次の作業

- `blocking_issues.csv`：止まった理由。`*_missing_games.csv`：追加で必要な対局IDと件数。
- `*_missingness.csv`：元特徴量のNaN。事象非発生の欠損は通常あり得ます。
- `*_same_hour.csv`：IDだけでは順序が決まらない同一時間内の対局。確認根拠が必要です。
- `fold_summary.csv`：両クラス、採点件数、窓・補完データ重複の確認。
- `external_input_summary_NO_PERFORMANCE.csv`：外部入力の件数・比率。性能結果ではありません。

各実行は新しいUTC時刻＋ランダムIDのフォルダへ保存し、既存ファイルを上書きしません。
外部評価は一度結果を見ているため完全な未使用テストではありません。今後の調整には使いません。
時系列窓同士の相関は評価区間内にも残ります。試合数を独立標本数と同一視しません。
両台帳の同一対局は照合します。相手プレイヤーの全登場履歴を別途分析する処理はありません。
本実装は実際の連続10試合を厳密に要求します。台帳や順序を保証できない場合、ランダム順や段位除外後の行列で代用しません。
